In [ ]:
!pip install transformers datasets torch

In [ ]:
import pandas as pd

df = pd.read_csv("merged_with_tweets_training.csv")
df = df[df["tweets"].notna() & (df["tweets"].str.strip() != "")]
df=df[["timestamp","tweets","congestion_level"]]
df

,timestamp,tweets,congestion_level
0,2025-02-22 03:30:00,"['Just drove to the store and it was a breeze,...",0
1,2024-11-09 09:15:00,"[""Just headed out for a run and it's a beautif...",0
2,2025-03-06 13:45:00,"[""Just braved the chilly morning commute, it's...",0
3,2024-04-04 15:30:00,"['Woke up to a pretty chilly grey day, 9°C out...",0
4,2025-01-08 21:15:00,"['Just got out of my car, finally parked in a ...",0
...,...,...,...
1278,2025-06-21 12:30:00,"[""I just saw someone blocking the bike lane on...",0
1279,2025-06-22 06:30:00,['Just got a parking ticket on Columbus Avenue...,0
1280,2025-07-08 09:30:00,"[""Driving down Ocean Parkway and there's a car...",0
1281,2025-07-08 09:30:00,"[""Just got a posted parking sign violation on ...",0


In [ ]:
import re

def clean(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = text.lower()
    return text

df["tweets"] = df["tweets"].apply(clean)
df

,timestamp,tweets,congestion_level
0,2025-02-22 03:30:00,"['just drove to the store and it was a breeze,...",0
1,2024-11-09 09:15:00,"[""just headed out for a run and it's a beautif...",0
2,2025-03-06 13:45:00,"[""just braved the chilly morning commute, it's...",0
3,2024-04-04 15:30:00,"['woke up to a pretty chilly grey day, 9°c out...",0
4,2025-01-08 21:15:00,"['just got out of my car, finally parked in a ...",0
...,...,...,...
1278,2025-06-21 12:30:00,"[""i just saw someone blocking the bike lane on...",0
1279,2025-06-22 06:30:00,['just got a parking ticket on columbus avenue...,0
1280,2025-07-08 09:30:00,"[""driving down ocean parkway and there's a car...",0
1281,2025-07-08 09:30:00,"[""just got a posted parking sign violation on ...",0


In [ ]:
import pandas as pd

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp")
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]
test_df

,timestamp,tweets,congestion_level
1202,2025-03-31 14:00:00,['just rode my bike near the nan street inters...,0
1203,2025-03-31 14:00:00,['just rode my bike past a traffic signal that...,0
1200,2025-03-31 14:00:00,['just rode my bike near the nan street inters...,0
856,2025-03-31 16:15:00,"[""really stuck in traffic today, 2 hours alrea...",2
896,2025-03-31 16:15:00,"[""hoping i don't hit too much traffic on the w...",2
...,...,...,...
1281,2025-07-08 09:30:00,"[""just got a posted parking sign violation on ...",0
338,2025-07-08 13:30:00,['just got stuck in a bit of traffic on the wa...,1
259,2025-07-08 13:30:00,"[""i'm loving the warm weather, perfect day to ...",0
419,2025-12-10 16:15:00,['just got stuck in the middle of town due to ...,1


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["tweets"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1026 [00:00<?, ? examples/s]

Map:   0%|          | 0/257 [00:00<?, ? examples/s]

In [ ]:
train_dataset = train_dataset.rename_column("congestion_level", "labels")
test_dataset = test_dataset.rename_column("congestion_level", "labels")

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./traffic_model",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    num_train_epochs=10
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.831859
1000,0.598846
1500,0.355155
2000,0.304501
2500,0.214832
3000,0.169197
3500,0.138551
4000,0.131403
4500,0.108709
5000,0.084087


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5130, training_loss=0.28930449736745734, metrics={'train_runtime': 33325.8955, 'train_samples_per_second': 0.308, 'train_steps_per_second': 0.154, 'total_flos': 1349771832944640.0, 'train_loss': 0.28930449736745734, 'epoch': 10.0})

In [ ]:
predictions = trainer.predict(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids
print("Accuracy:", accuracy_score(labels, preds))
print(classification_report(labels, preds))

Accuracy: 0.8326848249027238
              precision    recall  f1-score   support

           0       0.83      0.90      0.87       115
           1       0.76      0.66      0.71        73
           2       0.90      0.90      0.90        69

    accuracy                           0.83       257
   macro avg       0.83      0.82      0.82       257
weighted avg       0.83      0.83      0.83       257



In [ ]:
import torch
import torch.nn.functional as F
df = df.reset_index(drop=True)
predictions = trainer.predict(test_dataset)
logits = torch.tensor(predictions.predictions)
probs = torch.softmax(logits, dim=1)
confidence, predicted_class = torch.max(probs, dim=1)
df["predicted_label"] = predicted_class.numpy()
df["confidence"] = confidence.numpy()

In [ ]:
df.to_csv("predictions.csv", index=False)

In [ ]:
from google.colab import files
files.download("predictions.csv")